# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Display dataset name and description using the Dataset metadata object
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets and fields with their `@id`s.

In [ ]:
# List available record sets and their fields by @id
print("Record sets in the dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {record_set['@id']} | Name: {record_set.get('name', '')}")
    if 'fields' in record_set:
        for field in record_set['fields']:
            print(f"    - Field @id: {field['@id']} | Name: {field.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract all tabular record sets into separate Pandas DataFrames using their Croissant `@id`. Replace `@id` with the IDs you want to explore from the overview above.

In [ ]:
# Collect all tabular record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet {record_set_id}")
# Display columns of the first record set
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"\nColumns in RecordSet {first_id}:")
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data filtering, normalization, grouping, and transformations using only the field `@id`s.

Let's select a numeric field and demonstrate outlier filtering, normalization, and grouping.

In [ ]:
# -- Set these @ids based on output above for reproducibility --
# For example, assume first record set id is 'ds:protein-abundance', numeric field is 'ds:abundance', and group field is 'ds:protein_description'
# Update the values below with actual field @ids after running the overview cell.
record_set_id = record_set_ids[0]  # Use your desired record set @id
numeric_field_id = None
group_field_id = None

# Attempt to auto-detect first numeric column
df = dataframes[record_set_id]
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
# Attempt to auto-detect first categorical/text column for grouping (skipping numeric)
for col in df.columns:
    if col != numeric_field_id and pd.api.types.is_string_dtype(df[col]):
        group_field_id = col
        break

print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group field @id: {group_field_id}")

if numeric_field_id:
    # Remove outliers (e.g., values <= 0)
    filtered = df[df[numeric_field_id] > 0]
    print(f"Filtered records with {numeric_field_id} > 0: {filtered.shape[0]} out of {df.shape[0]}")

    # Normalize the numeric field
    filtered[f"{numeric_field_id}_normalized"] = (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
    print(filtered[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group and aggregate
    if group_field_id:
        grouped = filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} by {group_field_id}:")
        print(grouped.head())
    else:
        print("No suitable group field for aggregation found.")
else:
    print("No numeric field for EDA found in the DataFrame.")

## 5. Visualization
Visualize key distributions or relationships. Examples below use matplotlib for numeric field distribution and grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered[numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    if group_field_id:
        # Plot top 10 groups by mean
        top_groups = grouped.sort_values(numeric_field_id, ascending=False).head(10)
        plt.figure(figsize=(10, 5))
        sns.barplot(y=group_field_id, x=numeric_field_id, data=top_groups)
        plt.title(f'Top 10 {group_field_id} by mean {numeric_field_id}')
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset using the `mlcroissant` library, explored record sets and fields using their Croissant `@id` values, extracted tabular data for processing, filtered and normalized numeric columns, and visualized the data distribution and group statistics. 

Adjust analysis parameters as desired for deeper exploration or further processing for scientific workflows.